In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [1]:
df = pd.read_excel("Data_Science.xlsx")

display(df.head())

print("Dataset shape:", df.shape)

df.info()

print(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())

,Transaction_ID,Customer_Name,City,Department,Product,Quantity,Unit_Price,Revenue,Expense,Profit,Customer_Rating,Customer_Feedback,Employee_Performance_Score,Payment_Method,Purchase_Date
0,TXN00001,Maria Dominguez,Hyderabad,HR,Headphones,2,69623,139246,79676,59570,1,Poor,98,Net Banking,2024-10-23
1,TXN00002,Jasmine Wells,Mumbai,Sales,Keyboard,3,5478,16434,7198,9236,4,Excellent,53,Net Banking,2024-06-04
2,TXN00003,Sherri Brown,Delhi,Finance,Tablet,8,2629,21032,12831,8201,2,Average,76,Net Banking,2024-11-22
3,TXN00004,Dr. Lisa Garcia,Mumbai,IT,Mobile,5,49088,245440,131238,114202,2,Good,86,Card,2024-03-28
4,TXN00005,Kelly Carpenter,Delhi,IT,Smartwatch,10,17066,170660,104544,66116,4,Poor,90,UPI,2024-02-06


Dataset shape: (8000, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Transaction_ID              8000 non-null   object
 1   Customer_Name               8000 non-null   object
 2   City                        8000 non-null   object
 3   Department                  8000 non-null   object
 4   Product                     8000 non-null   object
 5   Quantity                    8000 non-null   int64 
 6   Unit_Price                  8000 non-null   int64 
 7   Revenue                     8000 non-null   int64 
 8   Expense                     8000 non-null   int64 
 9   Profit                      8000 non-null   int64 
 10  Customer_Rating             8000 non-null   int64 
 11  Customer_Feedback           8000 non-null   object
 12  Employee_Performance_Score  8000 non-null   int64 
 13  Payment_Method        

In [2]:
df["Purchase_Date"] = pd.to_datetime(
    df["Purchase_Date"],
    errors="coerce"
)

In [3]:
df['Customer_Feedback'] = (
    df['Customer_Feedback']
    .str.strip()
    .str.lower()
)

In [4]:
df['Customer_Feedback'].isnull().sum()

np.int64(0)

In [5]:
df['Customer_Feedback'].value_counts()

,count
Customer_Feedback,
average,2104
good,1995
excellent,1980
poor,1921


In [7]:
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(
    df['Customer_Feedback']
)

print("TF-IDF shape:", X_tfidf.shape)
print("Vocabulary:", tfidf.get_feature_names_out())

TF-IDF shape: (8000, 4)
Vocabulary: ['average' 'excellent' 'good' 'poor']


In [8]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=df.index
)

tfidf_df.head()

,average,excellent,good,poor
0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0


In [9]:
sentiment_mapping = {
    'poor': 'Negative',
    'average': 'Neutral',
    'good': 'Positive',
    'excellent': 'Positive'
}

df['Sentiment'] = (
    df['Customer_Feedback']
    .map(sentiment_mapping)
)

df[['Customer_Feedback', 'Sentiment']].head()

,Customer_Feedback,Sentiment
0,poor,Negative
1,excellent,Positive
2,average,Neutral
3,good,Positive
4,poor,Negative


In [12]:
X = df['Customer_Feedback']
y = df['Sentiment']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression())
])

In [16]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('classifier', LogisticRegression())])

In [17]:
y_pred = model.predict(X_test)

In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       384
     Neutral       1.00      1.00      1.00       421
    Positive       1.00      1.00      1.00       795

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600



In [30]:
from sklearn.metrics import accuracy_score

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Training Accuracy:",
      accuracy_score(y_train, y_train_pred))

print("Testing Accuracy:",
      accuracy_score(y_test, y_test_pred))

Training Accuracy: 1.0
Testing Accuracy: 1.0


In [34]:
# Predict sentiment for every row
df['Sentiment'] = model.predict(
    df['Customer_Feedback']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

# Display the last five columns
df.iloc[:, -5:].head()

,Customer_Feedback,Employee_Performance_Score,Payment_Method,Purchase_Date,Sentiment
0,poor,98,Net Banking,2024-10-23,Negative
1,excellent,53,Net Banking,2024-06-04,Positive
2,average,76,Net Banking,2024-11-22,Neutral
3,good,86,Card,2024-03-28,Positive
4,poor,90,UPI,2024-02-06,Negative


In [35]:
df.to_excel(
    'Data_Science_Sentiment_Analysis.xlsx',
    index=False
)

In [32]:
new_reviews = [
    "The product quality is excellent and delivery was fast",
    "The service was very poor and the product arrived damaged",
    "The product is okay but nothing special",
    "I expected better quality",
    "The food was not good"
]

predictions = model.predict(new_reviews)

for review, sentiment in zip(new_reviews, predictions):
    print(f"Review: {review}")
    print(f"Predicted Sentiment: {sentiment}\n")

Review: The product quality is excellent and delivery was fast
Predicted Sentiment: Positive

Review: The service was very poor and the product arrived damaged
Predicted Sentiment: Negative

Review: The product is okay but nothing special
Predicted Sentiment: Positive

Review: I expected better quality
Predicted Sentiment: Positive

Review: The food was not good
Predicted Sentiment: Positive

